Data loading and preprocessing copied from example notebook in Xplique library:
https://colab.research.google.com/drive/1iuEz46ZjgG97vTBH8p-vod3y14UETvVE#scrollTo=5oT_JAvEgjhv

In [1]:
import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [2]:
def central_crop_and_resize(img, input_size):

  h, w, _ = img.shape

  min_side = min(h, w)
  max_side_center = max(h, w) // 2.0

  min_cut = int(max_side_center-min_side//2)
  max_cut = int(max_side_center+min_side//2)

  img = img[:, min_cut:max_cut] if w > h else img[min_cut:max_cut]
  img = tf.image.resize(img, (input_size, input_size)).numpy()

  return img

In [3]:
import os
os.chdir('../data/Unsplash/images_optimize_k/')
!pwd

/mnt/KInsektDaten/data/Unsplash/images_optimize_k


In [ ]:
nb_samples = 100
input_size = 299

def load_collection(collection_name, nb_samples):
  !rm -rf $collection_name && mkdir $collection_name
  filenames = [f"{collection_name}/[{i}].jpg" for i in range(nb_samples)]
  for i in range(nb_samples):
    !curl -L --silent http://source.unsplash.com/featured?$collection_name -o "$collection_name/[$i].jpg"
    time.sleep(0.6)

  imgs = [cv2.imread(img)[...,::-1] for img in filenames]
  imgs = [central_crop_and_resize(img, input_size) for img in imgs]
  imgs = np.array(imgs)
  print(collection_name + ' downloaded.')
  return imgs

# indexes are from ImageNet and zero-based
classes_to_load = [
                   ('zebra',  340),
                   ('tiger',  292),
                   ('cheetah', 293),
                   ('snake', 57), # garter snake
                   ('turtle', 35) # mud turtle
]
classes_images = {(cls, id): load_collection(cls, nb_samples) for (cls, id) in classes_to_load}

In [ ]:
plt.rcParams["figure.figsize"] = [25, 10]
for (class_name, class_id), images in classes_images.items():
  print(f"\n\nClass: {class_name}  - shape: {images.shape}")
  for i in range(min(10, nb_samples)):
    plt.subplot(1, 10, i+1)
    plt.imshow(images[i] / 255.0)
    plt.axis('off')
  plt.show()